Data Cleaning and Exploration
Indicated damage is the class label we want to predict

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
missing_values = ["UNKNOWN", "Unknown"]
train_data = pd.read_csv('train.csv', na_values=missing_values, header='infer')
print("Number of rows and columns in training data: ", train_data.shape)
train_data.head()
train_data.info()

In [ ]:
train_data.describe()

In [ ]:
# Handling missing values
null_column_totals = train_data.isnull().sum().sort_values(ascending=False)
print("Number missing values for each column\n", null_column_totals)

In [ ]:
# remove any duplicates
print("Number of duplicate rows", train_data.duplicated().sum())
train_data.drop_duplicates()
# Shape of dataset remained the same, no duplicates found in training dataset

In [ ]:
# Drop columns that do not relate to prediction: remarks and comments
# Drop columns with less than 70% of row values filled
# BIRD_BAND_NUMBER        306465
# ENG_4_POS               303909
# ENROUTE_STATE           301832
# PRECIPITATION           295966
# ENG_3_POS               294822
# LOCATION                267961
# SPEED                   212076
# NUM_SEEN                208546
# WARNED                  191135
# SKY                     162825
# FLT                     161438
# HEIGHT                  154197
# TIME_OF_DAY             133971
# PHASE_OF_FLIGHT         120970
# REG                     118991
# AMO                     116813
# EMO                     113176
# TIME                    109252
# DISTANCE                103303
# COMMENTS                103083
# ENG_2_POS               102509
# EMA                     102421
print(train_data.shape)
# Drop columns with less than 70% of rows filled
train_data = train_data.drop(columns=["BIRD_BAND_NUMBER", "ENG_4_POS", "PRECIPITATION", "ENG_3_POS", "LOCATION", "SPEED", "NUM_SEEN", "WARNED", "SKY", "FLT", "HEIGHT", "TIME_OF_DAY", "PHASE_OF_FLIGHT", "REG", "AMO", "EMO", "TIME", "DISTANCE", "ENG_2_POS", "EMA"])
print(train_data.shape)


# Drop Airports since AIRPORT is just name corresponding to AIRPORT_ID (reduce dimensionality)
train_data = train_data.drop(columns=["AIRPORT"])
# Drop LOCATION or standardize: many null values, inconsistent data format: FOUND AT KATL; DRO-DEN; Found KMEM; Waldorf, MD, 15 MILES KADW
# train_data = train_data.drop(columns=["LOCATION"])
# Drop OPERATOR since OPERATOR is name that corresponds to OPID (reduce dimensionality)
train_data = train_data.drop(columns=["OPERATOR"])
# Drop PRECIPITATION: MANYYY Empty values (Imputation may lead to incorrect results)
# train_data = train_data.drop(columns=["PRECIPITATION"])
# Drop BIRD_BAND_ID: Many empty values (Imputation may lead to incorrect results)
# train_data = train_data.drop(columns=["BIRD_BAND_ID"])
# Drop SPECIES: SPECIES_ID is corresponding ID for common name (SPECIES)
train_data = train_data.drop(columns=["SPECIES"])
# COMMENTS: just notes
train_data = train_data.drop(columns=["COMMENTS"])
# REMARKS: just notes
train_data = train_data.drop(columns=["REMARKS"])
# Drop Source: Type of report filed, may not be needed to predict indicated damage
train_data = train_data.drop(columns=["SOURCE"])
# Drop Person: Role of person who filed report, dont need for prediction
train_data = train_data.drop(columns=["PERSON"])

# NOTE: we found that STATE and FAAREGION were correlated and when one was null, so was the other
# decided to remove FAAREGION 
state = train_data["STATE"].isnull()
faaregion = train_data["FAAREGION"].isnull()
shared_null = (state & faaregion).sum()
print(shared_null)
train_data = train_data.drop(columns=["FAAREGION"])


# FOR ENROUTE STATE: corresponding state column is typically empty, fill with ENROUTE STATE if present

Data Exploration: Graphs

In [ ]:
# This will display the distribution of the target variable, whether wildlife strike caused damage
# NOTES: this is an imbalanced distribution, we should not use accuracy as a metric and instead use confusion matrices
# With precision and recall to determine error
train_data["INDICATED_DAMAGE"].value_counts().plot.bar()

In [ ]:
# Display the number of strikes based on state
train_data["STATE"].value_counts()

Determining Outliers using Box Plots

In [ ]:
numeric_columns = train_data.select_dtypes(include='number')
numeric_columns.plot(kind='box', subplots=True, layout=(10, 5), figsize=(20,20))

Handling Missing Values

In [ ]:
# Drop records that are missing NUM_STRUCK
data_pre_drop = train_data.shape[0]
train_data.dropna(axis=0, subset=['NUM_STRUCK'], inplace=True)
data_post_drop = train_data.shape[0]
num_dropped = data_pre_drop - data_post_drop
print("Dropped ", num_dropped, " records\n")


In [ ]:
# Handling missing values
null_column_totals = train_data.isnull().sum().sort_values(ascending=False)
null_column_totals

In [ ]:
# Imputation for LATITUDE and LOGITUDE
# We found when imputing that the latitude and longitude has some errors in the data resulting in some entries to have commas which prevents us from calculating the mean
# Solution: drop records that contain invalid format
dms_count = train_data['LATITUDE'].astype(str).str.contains("°", na=False).sum()
total = len(train_data)
print(dms_count)

dms_count = train_data['LONGITUDE'].astype(str).str.contains("°", na=False).sum()
total_longitude = len(train_data)
print(dms_count)

# Drop records with latitude or longitude in DMS format
print(len(train_data))
train_data = train_data[~train_data['LATITUDE'].astype(str).str.contains("°", na=False)]
train_data = train_data[~train_data['LONGITUDE'].astype(str).str.contains("°", na=False)]
print(len(train_data))

train_data["LATITUDE"] = train_data["LATITUDE"].astype(str).str.replace(",", "")
train_data["LONGITUDE"] = train_data["LONGITUDE"].astype(str).str.replace(",", "")
train_data["LATITUDE"] = pd.to_numeric(train_data["LATITUDE"], errors="coerce")
train_data["LONGITUDE"] = pd.to_numeric(train_data["LONGITUDE"], errors="coerce")

print("Number of null values before imputation")
print(train_data["LATITUDE"].isnull().sum())
print(train_data["LONGITUDE"].isnull().sum())

train_data['LATITUDE'] = train_data['LATITUDE'].fillna(train_data['LATITUDE'].mean())
train_data['LONGITUDE'] = train_data['LONGITUDE'].fillna(train_data['LONGITUDE'].mean())

print("After imputation, check that no null values remain")
print(train_data["LATITUDE"].isnull().sum())
print(train_data["LONGITUDE"].isnull().sum())

In [ ]:
# If STATE is null, impute using ENROUTE_STATE, then remove ENROUTE_STATE column in the end
# Make a mask to show the records that enroute state has values for
# Make a mask for the records that STATE is null in
# Compute intersection between masks to get records that will be imputed with ENROUTE_STATE
# Fill missing STATE using ENROUTE_STATE
print("Pre fill: ", train_data["STATE"].isnull().sum())
train_data['STATE'] = train_data['STATE'].fillna(train_data['ENROUTE_STATE'])
print("Post fill: ", train_data["STATE"].isnull().sum())

# Drop ENROUTE_STATE
train_data = train_data.drop(columns=['ENROUTE_STATE'])

# Similar case imputation based on AIRPORT_ID since there is no null values present
print("Null entries after imputation", train_data["STATE"].isnull().sum(), "\n")
airport_states = train_data.groupby("AIRPORT_ID")["STATE"].transform("first")
train_data["STATE"] = train_data["STATE"].fillna(airport_states)
print("Null entries after imputation", train_data["STATE"].isnull().sum(), "\n")

# We have 11 remaining null entries, we can then drop those records with null states
train_data = train_data.dropna(subset=["STATE"])
print("Null entries after dropping remainder", train_data["STATE"].isnull().sum(), "\n")

# Standardize foreign STATE to only have one value (currently FN/FGN)
train_data["STATE"] = train_data["STATE"].replace("FN", "FGN")

Handling Categorical Features: One Hot Encoding

In [ ]:
# need to one hot encode categorical variables
# categorical_columns =['NUM_STRUCK', 'TIME_OF_DAY', 'AIRPORT_ID', 'RUNWAY', 'STATE', 'FAAREGION', 'LOCATION', 'OPID', 'AIRCRAFT', 'AC_CLASS', 'TYPE_ENG', 'PHASE_OF_FLIGHT', 'SKY', 'SPECIES_ID', 'WARNED', 'SIZE', 'ENROUTE_STATE', 'SOURCE', 'PERSON']
categorical_columns =['NUM_STRUCK', 'AIRPORT_ID', 'RUNWAY', 'STATE', 'FAAREGION', 'OPID', 'AIRCRAFT', 'AC_CLASS', 'TYPE_ENG', 'SPECIES_ID', 'SIZE']
train_data = pd.get_dummies(train_data, columns = categorical_columns, drop_first=True)


# WHAT TO ONE HOT ENCODE
# NUM_STRUCK, TIME_OF_DAY, AIRPORT_ID, RUNWAY, STATE, FAAREGION, LOCATION, OPID, AIRCRAFT, AC_CLASS, TYPE_ENG, PHASE_OF_FLIGHT, SKY, SPECIES_ID, 
# WARNED, SIZE, ENROUTE_STATE, SOURCE, PERSON

train_data.head()